In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
# Does explanation order matter? 
# Experiment 1: Compare across ALL models
# Experiment 2: Compare just Gemma (which has answer_first variation)

In [3]:
# =============================================================================
# EXPERIMENT 1: All models
# =============================================================================
families_all = ['gpt_5_medium', 'claude_4_5_medium', 'gemini_3_medium', 'qwen_3', 'gemma_3']
all_data_all = []
for f in families_all:
    df_ = pd.read_parquet(f"../experiments/combined/{f}/predictions.parquet")
    df_['family'] = f
    all_data_all.append(df_)

combined_df_all = pd.concat(all_data_all)

# Filter out rows with None answers
answer_cols = ['original_reference_response_answer', 'counterfactual_reference_response_answer', 
               'counterfactual_predictor_response_with_explanation_answer', 
               'counterfactual_predictor_response_without_explanation_answer']
filtered_df_all = combined_df_all.dropna(subset=answer_cols)

print(f"Total records: {len(filtered_df_all)}")
print(f"Answer first True: {filtered_df_all['original_answer_first'].sum()}")
print(f"Answer first False: {(~filtered_df_all['original_answer_first']).sum()}")

Total records: 122205
Answer first True: 60706
Answer first False: 61499


In [4]:
# Clustered bootstrapping function (with multiple predictors exploded)

def bootstrap_nsg_by_answer_order(df, n_bootstrap=1000, confidence=0.95, cluster_col='original_question_idx', debug=False):
    """
    Clustered bootstrap for normalized simulatability gain by answer order.
    
    Explodes predictor_answers lists so each predictor is a separate observation.
    Resamples by cluster (original question) to handle correlation.
    """
    np.random.seed(42)

    df = df.copy().reset_index(drop=True)
    
    # Explode predictor_answers - each predictor becomes a separate row
    exploded_rows = []
    skipped_none_lists = 0
    skipped_none_individual = 0
    
    for idx, row in df.iterrows():
        with_answers = row['counterfactual_predictor_response_with_explanation_predictor_answers']
        without_answers = row['counterfactual_predictor_response_without_explanation_predictor_answers']
        ref_answer = row['counterfactual_reference_response_answer']
        answer_first = row['original_answer_first']
        q_idx = row[cluster_col]
        
        # Skip if lists are None or empty
        if with_answers is None or without_answers is None:
            skipped_none_lists += 1
            continue
            
        n_predictors = min(len(with_answers), len(without_answers))
        for i in range(n_predictors):
            if with_answers[i] is not None and without_answers[i] is not None:
                exploded_rows.append({
                    'cluster': q_idx,
                    'answer_first': answer_first,
                    'with_correct': with_answers[i] == ref_answer,
                    'without_correct': without_answers[i] == ref_answer,
                })
            else:
                skipped_none_individual += 1
    
    exploded_df = pd.DataFrame(exploded_rows)
    
    if debug:
        print(f"Debug info:")
        print(f"  Input rows: {len(df)}")
        print(f"  Skipped (None predictor_answers list): {skipped_none_lists}")
        print(f"  Skipped (None individual predictor answer): {skipped_none_individual}")
        print(f"  Exploded observations: {len(exploded_df)}")
        print(f"  Unique clusters: {exploded_df['cluster'].nunique()}")
        print(f"  Observations per answer_first group:")
        for val in exploded_df['answer_first'].unique():
            count = (exploded_df['answer_first'] == val).sum()
            print(f"    answer_first={val}: {count}")
        print()
    
    # Pre-compute cluster -> row indices mapping
    cluster_to_rows = exploded_df.groupby('cluster').indices
    clusters = np.array(list(cluster_to_rows.keys()))
    n_clusters = len(clusters)
    
    # Pre-extract columns as numpy arrays
    answer_first = exploded_df['answer_first'].values
    with_correct = exploded_df['with_correct'].values
    without_correct = exploded_df['without_correct'].values
    unique_groups = exploded_df['answer_first'].unique()

    # Point estimate
    point_est = exploded_df.groupby('answer_first')[['with_correct', 'without_correct']].mean()
    point_est['nsg'] = (point_est['with_correct'] - point_est['without_correct']) / (1 - point_est['without_correct'])

    if debug:
        print("Point estimates (before bootstrap):")
        print(point_est)
        print()

    # Bootstrap storage
    boot_results = {group: [] for group in unique_groups}

    for _ in tqdm(range(n_bootstrap)):
        # Resample cluster indices
        sampled_cluster_indices = np.random.randint(0, n_clusters, size=n_clusters)
        sampled_clusters = clusters[sampled_cluster_indices]
        
        # Get all row indices for sampled clusters
        row_indices = np.concatenate([cluster_to_rows[c] for c in sampled_clusters])
        
        # Extract values using indices
        boot_groups = answer_first[row_indices]
        boot_with = with_correct[row_indices]
        boot_without = without_correct[row_indices]
        
        # Calculate stats per group
        for group in unique_groups:
            mask = boot_groups == group
            if mask.sum() > 0:
                mean_with = boot_with[mask].mean()
                mean_without = boot_without[mask].mean()
                nsg = (mean_with - mean_without) / (1 - mean_without) if mean_without < 1 else 0
                boot_results[group].append(nsg)

    # Compute CIs
    alpha = 1 - confidence
    result = point_est.copy()
    result['nsg_ci_lower'] = [np.percentile(boot_results[g], alpha/2 * 100) for g in result.index]
    result['nsg_ci_upper'] = [np.percentile(boot_results[g], (1 - alpha/2) * 100) for g in result.index]

    return result

In [5]:
# Run bootstrap on ALL models (with debug info)
print("Running bootstrap on all models...")
bootstrap_results_all = bootstrap_nsg_by_answer_order(filtered_df_all, debug=True)
bootstrap_results_all

Running bootstrap on all models...
Debug info:
  Input rows: 122205
  Skipped (None predictor_answers list): 0
  Skipped (None individual predictor answer): 4
  Exploded observations: 366611
  Unique clusters: 3656
  Observations per answer_first group:
    answer_first=True: 182115
    answer_first=False: 184496

Point estimates (before bootstrap):
              with_correct  without_correct       nsg
answer_first                                         
False             0.782115         0.669527  0.340687
True              0.779376         0.664349  0.342696



100%|██████████| 1000/1000 [00:03<00:00, 259.45it/s]


,with_correct,without_correct,nsg,nsg_ci_lower,nsg_ci_upper
answer_first,,,,,
False,0.782115,0.669527,0.340687,0.328244,0.354562
True,0.779376,0.664349,0.342696,0.329670,0.354592


In [6]:
# =============================================================================
# EXPERIMENT 2: Gemma only (has balanced answer_first True/False)
# =============================================================================
families_gemma = ['gemma_3']
all_data_gemma = []
for f in families_gemma:
    df_ = pd.read_parquet(f"../experiments/combined/{f}/predictions.parquet")
    df_['family'] = f
    all_data_gemma.append(df_)

combined_df_gemma = pd.concat(all_data_gemma)

# Filter out rows with None answers
filtered_df_gemma = combined_df_gemma.dropna(subset=answer_cols)

print(f"Gemma records: {len(filtered_df_gemma)}")
print(f"Answer first True: {filtered_df_gemma['original_answer_first'].sum()}")
print(f"Answer first False: {(~filtered_df_gemma['original_answer_first']).sum()}")

Gemma records: 26618
Answer first True: 12925
Answer first False: 13693


In [7]:
# Run bootstrap on Gemma only (with debug info)
print("Running bootstrap on Gemma...")
bootstrap_results_gemma = bootstrap_nsg_by_answer_order(filtered_df_gemma, debug=True)
bootstrap_results_gemma

Running bootstrap on Gemma...
Debug info:
  Input rows: 26618
  Skipped (None predictor_answers list): 0
  Skipped (None individual predictor answer): 0
  Exploded observations: 79854
  Unique clusters: 2472
  Observations per answer_first group:
    answer_first=True: 38775
    answer_first=False: 41079

Point estimates (before bootstrap):
              with_correct  without_correct       nsg
answer_first                                         
False             0.778646         0.654349  0.359603
True              0.784500         0.655371  0.374691



100%|██████████| 1000/1000 [00:01<00:00, 806.00it/s]


,with_correct,without_correct,nsg,nsg_ci_lower,nsg_ci_upper
answer_first,,,,,
False,0.778646,0.654349,0.359603,0.338605,0.378872
True,0.784500,0.655371,0.374691,0.354797,0.396911


In [8]:
# Compare results
print("=" * 60)
print("SUMMARY: Does explanation order matter?")
print("=" * 60)
print("\nAll Models:")
print(f"  Answer First (True):  NSG = {bootstrap_results_all.loc[True, 'nsg']*100:.1f}% ({bootstrap_results_all.loc[True, 'nsg_ci_lower']*100:.1f}%, {bootstrap_results_all.loc[True, 'nsg_ci_upper']*100:.1f}%)")
print(f"  Answer Second (False): NSG = {bootstrap_results_all.loc[False, 'nsg']*100:.1f}% ({bootstrap_results_all.loc[False, 'nsg_ci_lower']*100:.1f}%, {bootstrap_results_all.loc[False, 'nsg_ci_upper']*100:.1f}%)")
print(f"  Difference: {(bootstrap_results_all.loc[True, 'nsg'] - bootstrap_results_all.loc[False, 'nsg'])*100:.2f}%")

print("\nGemma Only:")
print(f"  Answer First (True):  NSG = {bootstrap_results_gemma.loc[True, 'nsg']*100:.1f}% ({bootstrap_results_gemma.loc[True, 'nsg_ci_lower']*100:.1f}%, {bootstrap_results_gemma.loc[True, 'nsg_ci_upper']*100:.1f}%)")
print(f"  Answer Second (False): NSG = {bootstrap_results_gemma.loc[False, 'nsg']*100:.1f}% ({bootstrap_results_gemma.loc[False, 'nsg_ci_lower']*100:.1f}%, {bootstrap_results_gemma.loc[False, 'nsg_ci_upper']*100:.1f}%)")
print(f"  Difference: {(bootstrap_results_gemma.loc[True, 'nsg'] - bootstrap_results_gemma.loc[False, 'nsg'])*100:.2f}%")

SUMMARY: Does explanation order matter?

All Models:
  Answer First (True):  NSG = 34.3% (33.0%, 35.5%)
  Answer Second (False): NSG = 34.1% (32.8%, 35.5%)
  Difference: 0.20%

Gemma Only:
  Answer First (True):  NSG = 37.5% (35.5%, 39.7%)
  Answer Second (False): NSG = 36.0% (33.9%, 37.9%)
  Difference: 1.51%


In [9]:
# Simple non-exploded calculation (for comparison)
# This uses the aggregated _answer columns instead of exploding predictor_answers
print("=" * 60)
print("SIMPLE NON-EXPLODED CALCULATION (for comparison)")
print("=" * 60)
print("This shows what you'd get if you use the _answer columns directly,")
print("instead of exploding the _predictor_answers lists.\n")

for name, df in [("All Models", filtered_df_all), ("Gemma Only", filtered_df_gemma)]:
    df = df.copy()
    df['with_correct'] = df['counterfactual_predictor_response_with_explanation_answer'] == df['counterfactual_reference_response_answer']
    df['without_correct'] = df['counterfactual_predictor_response_without_explanation_answer'] == df['counterfactual_reference_response_answer']
    
    simple_results = df.groupby('original_answer_first')[['with_correct', 'without_correct']].mean()
    simple_results['nsg'] = (simple_results['with_correct'] - simple_results['without_correct']) / (1 - simple_results['without_correct'])
    
    print(f"{name}:")
    for answer_first_val in [True, False]:
        if answer_first_val in simple_results.index:
            row = simple_results.loc[answer_first_val]
            print(f"  answer_first={answer_first_val}: with={row['with_correct']:.4f}, without={row['without_correct']:.4f}, NSG={row['nsg']*100:.1f}%")
    print()

SIMPLE NON-EXPLODED CALCULATION (for comparison)
This shows what you'd get if you use the _answer columns directly,
instead of exploding the _predictor_answers lists.

All Models:
  answer_first=True: with=0.7768, without=0.6665, NSG=33.1%
  answer_first=False: with=0.7790, without=0.6719, NSG=32.6%

Gemma Only:
  answer_first=True: with=0.7702, without=0.6525, NSG=33.9%
  answer_first=False: with=0.7548, without=0.6399, NSG=31.9%

